In [ ]:
# Instala dependências e conecta o Google Drive
# Instala as bibliotecas mais recentes do Hugging Face
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Monta o Drive para salvar arquivos de forma segura
drive.mount('/content/drive')

# Cria o diretório do projeto no Drive se ele não existir
PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PASTA_PROJETO, exist_ok=True)
print(f"Diretório de trabalho pronto em: {PASTA_PROJETO}")

Mounted at /content/drive
Diretório de trabalho pronto em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


In [ ]:
# Install dependencies and connect Google Drive
# Install the latest Hugging Face libraries
!pip install -q transformers datasets accelerate tokenizers

from google.colab import drive
import os

# Mount Drive to save files securely
drive.mount('/content/drive')

# Create the project directory in Drive if it does not exist
PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
os.makedirs(PASTA_PROJETO, exist_ok=True)
print(f"Diretório de trabalho pronto em: {PASTA_PROJETO}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Diretório de trabalho pronto em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite


## Configuração do Token do Hugging Face Hub

Para evitar avisos de requisições não autenticadas e potencialmente acelerar os downloads de modelos e datasets, é recomendável configurar um token de acesso do Hugging Face Hub. Siga os passos abaixo:

1.  **Obtenha seu Token de Acesso:**
    *   Vá para [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
    *   Crie um novo token. Recomenda-se um token com permissão `read` (leitura) para a maioria das operações de download.

2.  **Adicione o Token aos Segredos do Colab:**
    *   No painel esquerdo do Google Colab, clique no ícone de chave (🔑) para abrir a interface de "Segredos".
    *   Clique em "Adicionar novo segredo".
    *   No campo "Nome", digite `HF_TOKEN`.
    *   No campo "Valor", cole o token que você obteve do Hugging Face.
    *   Certifique-se de ativar a opção "Acesso ao notebook" para que o notebook possa usar este segredo.

3.  **Execute a célula Python abaixo:**
    *   Esta célula carregará o token dos segredos do Colab e o configurará como uma variável de ambiente, que será usada pelas bibliotecas do Hugging Face.

In [ ]:
# Importe as bibliotecas necessárias
from google.colab import userdata
import os
from huggingface_hub import login # Importa a função login do Hugging Face Hub

# Carregue o token do Hugging Face dos segredos do Colab
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        # Tenta fazer o login com o token do Hugging Face
        login(token=hf_token, add_to_git_credential=False) # 'add_to_git_credential=False' para não solicitar credenciais git
        print("Token do Hugging Face carregado e configurado com sucesso via huggingface_hub.login().")
    else:
        print("AVISO: O segredo 'HF_TOKEN' foi encontrado, mas está vazio ou é None. Por favor, verifique o valor nos segredos do Colab.")
        # Se o token estiver vazio/None, ainda define a variável de ambiente como string vazia para evitar erros posteriores
        os.environ['HF_TOKEN'] = ''
except userdata.SecretNotFoundError:
    print("ATENÇÃO: O segredo 'HF_TOKEN' não foi encontrado. Por favor, adicione seu token do Hugging Face aos segredos do Colab.")
    os.environ['HF_TOKEN'] = '' # Garante que a variável de ambiente seja definida, mesmo que vazia
except Exception as e:
    print(f"Ocorreu um erro ao carregar ou configurar o token do Hugging Face: {e}")
    os.environ['HF_TOKEN'] = '' # Garante que a variável de ambiente seja definida, mesmo que vazia

Token do Hugging Face carregado e configurado com sucesso via huggingface_hub.login().


In [ ]:
from datasets import load_dataset, concatenate_datasets
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2TokenizerFast
import os
import shutil

PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
pasta_tok = os.path.join(PASTA_PROJETO, "tokenizador")

# Verifica se o tokenizador já existe e o carrega, caso contrário, treina um novo.
if os.path.exists(pasta_tok) and os.path.isdir(pasta_tok) and \
   os.path.exists(os.path.join(pasta_tok, "vocab.json")) and \
   os.path.exists(os.path.join(pasta_tok, "merges.txt")):
    print(f"Tokenizador encontrado em: {pasta_tok}. Carregando tokenizador existente...")
    tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, local_files_only=True)
    print("Tokenizador carregado com sucesso!")
else:
    print(f"Tokenizador não encontrado ou incompleto em {pasta_tok}. Treinando novo tokenizador...")

    # 1. Carrega frações exatas direto para o cache local do Colab (SEM streaming=True)
    # Dividido proporcionalmente para somar 50.000 artigos no total (50% EN, 25% PT, 25% ES)
    print("Baixando fatias da Wikipédia para a memória local (Isso leva cerca de 1-2 minutos)...")
    wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train[:25000]")
    wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train[:12500]")
    wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train[:12500]")

    # Junta os datasets baixados localmente e embaralha na memória RAM
    print("Misturando e preparando os dados...")
    dataset_misto = concatenate_datasets([wiki_en, wiki_pt, wiki_es])
    dataset_misto = dataset_misto.shuffle(seed=42)

    # Gerador usado para alimentar o treinador do tokenizador direto da memória RAM
    def extrair_texto():
        for item in dataset_misto:
            yield item["text"]

    print("Treinando o tokenizador... Agora deve levar de 2 a 3 minutos.")
    tokenizer_raw = ByteLevelBPETokenizer()
    tokenizer_raw.train_from_iterator(
        extrair_texto(),
        vocab_size=50257, # Padrão clássico do GPT-2
        min_frequency=2,
        special_tokens=["<s>", "<pad>", "</s>", "<unk>", "<mask>"]
    )

    # Salva o tokenizador no Drive
    # REMOVE o diretório existente antes de criar novamente para forçar a sincronização
    if os.path.exists(pasta_tok) and os.path.isdir(pasta_tok):
        print(f"Removendo diretório existente do tokenizador: {pasta_tok}")
        shutil.rmtree(pasta_tok)

    os.makedirs(pasta_tok, exist_ok=True) # Cria o diretório novamente
    tokenizer_raw.save_model(pasta_tok)

    # Converte para o formato utilizável pelo Hugging Face Trainer
    tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, bos_token="<s>", eos_token="</s>", unk_token="<unk>", pad_token="<pad>", mask_token="<mask>")
    tokenizer.save_pretrained(pasta_tok)
    print(f"Tokenizador salvo com sucesso em: {pasta_tok}")

Tokenizador encontrado em: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/tokenizador. Carregando tokenizador existente...
Tokenizador carregado com sucesso!


In [ ]:

import os
import torch
import glob
from datasets import load_dataset, interleave_datasets
from transformers import GPT2Config, GPT2LMHeadModel, GPT2TokenizerFast, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from transformers.trainer_utils import get_last_checkpoint

# Adiciona esta linha para otimização de memória do PyTorch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

PASTA_PROJETO = "/content/drive/MyDrive/Colab_LLMs/Maia_Lite"
pasta_tok = os.path.join(PASTA_PROJETO, "tokenizador")
pasta_saida = os.path.join(PASTA_PROJETO, "checkpoints_pretreino")

# VERIFICAÇÃO DE GPU DINÂMICA
if torch.cuda.is_available():
    nome_gpu = torch.cuda.get_device_name(0)
    print(f"GPU disponível: {nome_gpu}")
    device = torch.device("cuda")
else:
    nome_gpu = "CPU"
    print("ATENÇÃO: Nenhuma GPU disponível. O treinamento será executado na CPU, o que será muito mais lento.")
    device = torch.device("cpu")

# 1. Reload the tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained(pasta_tok, local_files_only=True)

# 2. Localizar o checkpoint mais recente usando lógica numérica da Hugging Face
ultimo_checkpoint = get_last_checkpoint(pasta_saida)

if ultimo_checkpoint is not None:
    print(f"Encontrado progresso anterior numérico. O Trainer irá carregar os pesos e o estado de: {ultimo_checkpoint}")
    # Carrega a estrutura de pesos já treinada anteriormente
    model = GPT2LMHeadModel.from_pretrained(ultimo_checkpoint)
else:
    print("Nenhum checkpoint anterior encontrado. Inicializando modelo com pesos aleatórios do zero.")
    config = GPT2Config(
        vocab_size=tokenizer.vocab_size,
        n_positions=1024,
        n_ctx=1024,
        n_embd=1024,
        n_layer=24,
        n_head=16,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=False,
    )
    model = GPT2LMHeadModel(config)

model.to(device)
print(f"Total de parâmetros do modelo: {model.num_parameters():,}")

# Ativa o gradient checkpointing para economizar memória
model.gradient_checkpointing_enable()

# 3. Load the datasets in streaming mode
wiki_en = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)
wiki_pt = load_dataset("wikimedia/wikipedia", "20231101.pt", split="train", streaming=True)
wiki_es = load_dataset("wikimedia/wikipedia", "20231101.es", split="train", streaming=True)
dataset_misto = interleave_datasets([wiki_en, wiki_pt, wiki_es], probabilities=[0.5, 0.25, 0.25], seed=42)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=1024)

# Dynamic mapping because of streaming
tokenized_dataset = dataset_misto.map(tokenize_function, batched=True, remove_columns=["id", "url", "title", "text"])
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

# --- PERFIL DE CONFIGURAÇÃO INTELIGENTE (ADAPTADO PARA L4 / A100) ---
# Valores padrão seguros para T4 antiga
batch_por_dispositivo = 1
acumulacao_passos = 16
usar_bf16 = False
usar_fp16 = True

if "A100" in nome_gpu:
    batch_por_dispositivo = 8    # Aproveita os 40GB/80GB da A100
    acumulacao_passos = 4        # Lote global = 32
    usar_bf16 = True             # Estabilidade máxima em hardware premium
    usar_fp16 = False
elif "L4" in nome_gpu:
    batch_por_dispositivo = 4    # Ajuste ideal para os 24GB da L4
    acumulacao_passos = 8        # Lote global equilibrado em 32 (4x8)
    usar_bf16 = True             # A L4 aceita BF16 nativo! Sem erros de scaler
    usar_fp16 = False
elif "V100" in nome_gpu:
    batch_por_dispositivo = 4
    acumulacao_passos = 8
    usar_bf16 = False
    usar_fp16 = True

# 4. Training Parameters Optimized for High-End/Cost-Effective GPUs
training_args = TrainingArguments(
    output_dir=pasta_saida,
    max_steps=300000,              # Meta de passos para atingir alta cobertura de tokens
    per_device_train_batch_size=batch_por_dispositivo,
    gradient_accumulation_steps=acumulacao_passos,
    save_steps=2000,               # Salva no Drive regularmente
    save_total_limit=2,            # Mantém apenas os dois checkpoints mais recentes
    logging_steps=100,             # Acompanhamento ágil das métricas no painel do Colab
    bf16=usar_bf16,
    fp16=usar_fp16,
    learning_rate=4e-4,
    weight_decay=0.01,
    warmup_steps=3000,
    dataloader_num_workers=2,      # Otimiza o fluxo de dados liberando a CPU do gargalo
    ignore_data_skip=True,         # Evita travamentos longos ao tentar pular dados já vistos no streaming mode
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print(f"Configuração aplicada para {nome_gpu}. Iniciando loop de treinamento...")

# EXECUÇÃO INTELIGENTE DO TREINAMENTO (COM OU SEM RETOMADA)
if ultimo_checkpoint is not None:
    print(f"Retomando o loop de treinamento do Trainer a partir de: {ultimo_checkpoint}")
    trainer.train(resume_from_checkpoint=ultimo_checkpoint)
else:
    print("Nenhum checkpoint para o Trainer restaurar. Iniciando loop do zero.")
    trainer.train()

# Save the consolidated final model
model.save_pretrained(os.path.join(PASTA_PROJETO, "modelo_335M_final"))
print("PRÉ-TREINO CONCLUÍDO COM SUCESSO!")

GPU disponível: NVIDIA L4
Encontrado progresso anterior numérico. O Trainer irá carregar os pesos e o estado de: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-12000


Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

Total de parâmetros do modelo: 354,823,168


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

Configuração aplicada para NVIDIA L4. Iniciando loop de treinamento...
Retomando o loop de treinamento do Trainer a partir de: /content/drive/MyDrive/Colab_LLMs/Maia_Lite/checkpoints_pretreino/checkpoint-12000


[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


Step,Training Loss
12100,3.578767
12200,3.571608


Step,Training Loss
12100,3.578767
12200,3.571608
